# Lab 3 exercise solution - part 2

Exercise: *Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.*

My "business area" here is teaching: this is a friendly assistant for absolute beginners of agentic AI. It answers questions like *what is an agent?* or *what are tools?* based on my own notes from the course (`agentic_ai_foundations_theory_mcp.pdf`), and it politely says so when a topic is not covered by the notes. If a visitor wants to get in touch to chat about agentic AI, the assistant records their email address with a tool - using the same hand-cranked agent loop as in lab 3.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

## The knowledge base: my course notes

The assistant's material is a single PDF with my notes from the course, sitting next to this notebook. We extract the text just like we did with the twin profile PDFs.

In [ ]:
reader = PdfReader("agentic_ai_foundations_theory_mcp.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(f"Loaded {len(reader.pages)} pages, {len(notes)} characters of notes")

## The system prompt

Compared to the digital twin, the role changes completely: this is not an assistant representing a person, but a patient tutor for absolute beginners. Three things matter here:

1. The tone: friendly, encouraging, no jargon without explanation.
2. The grounding rule: answers must come from the notes, and if the notes don't cover a topic, the assistant must politely say so instead of making something up.
3. The tool instruction: if a visitor wants to get in touch, ask for their email address and record it with the tool.

In [ ]:
system_prompt = f"""

# Your role

You are a friendly and patient assistant for learners of agentic AI, running on a website.
Your visitors are absolute beginners who want to understand the basics: what agents are, what tools are,
how LLMs fit into the picture, what MCP is, and so on.

# Your style

Assume no prior knowledge of agentic AI. Explain concepts in simple, everyday language and
introduce technical terms gently, always explaining them first.
Keep your answers short and clear, and prefer a concrete example over an abstract definition.
Be warm and encouraging - your visitors are just starting out, and learning something new takes courage.

# Your material

Base your answers on the following course notes:

{notes}

# Rules

IMPORTANT: Only answer based on the course notes above. If a question cannot be answered from the notes,
politely say that this topic is not covered in your material - never make up an answer.
When you have to decline, be kind about it and suggest a related topic from the notes that the visitor could ask about instead.

If a question is not about agentic AI at all, politely steer the conversation back to agentic AI.

If the visitor would like to get in touch to chat about agentic AI in general, ask for their email address
and record it using your record_email_tool. Confirm to the visitor that their email has been recorded.
"""

## The email registration tool

Reused from lab 3: a plain Python function that appends the visitor's email address to a text file, and the JSON description that tells the LLM what the tool does and what arguments it expects.

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record the email address of a visitor who wants to get in touch to chat about agentic AI",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this visitor"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": record_email_tool_json}]

## The chat function with the agent loop

The same agent loop as in step 3 of lab 3: as long as the model wants to call tools, we execute each requested tool call, append the results to the conversation, and call the model again. When the model has nothing more to ask of the tools, it produces the final reply for the visitor.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            result = record_email_tool(email)
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    return response.choices[0].message.content

## Let's try it

Three test cases before launching the UI:

1. A beginner question that the notes cover - should get a friendly, simple answer.
2. A question the notes don't cover - should get a polite "that's not in my material" with a suggestion.
3. A visitor sharing their email - should trigger the tool (watch for the printed line) and land in `emails.txt`.

In [ ]:
display(Markdown(chat("I'm completely new to this. What is an AI agent?", [])))

In [ ]:
display(Markdown(chat("How do I fine-tune my own LLM from scratch?", [])))

In [ ]:
display(Markdown(chat("This was really helpful! I'd love to chat more about agentic AI - you can reach me at test@testy.com", [])))

And finally the chat UI:

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)